# 04 — Sampling strategies — independent vs LHS vs MVN

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-systems-today/cubespec/blob/main/notebooks/04_sampling_strategies.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ai-systems-today/cubespec/main?urlpath=lab/tree/notebooks/04_sampling_strategies.ipynb) [![nbviewer](https://img.shields.io/badge/render-nbviewer-orange)](https://nbviewer.org/github/ai-systems-today/cubespec/blob/main/notebooks/04_sampling_strategies.ipynb) [![Dashboard](https://img.shields.io/badge/open-dashboard-2563eb)](https://sensitive-spark.lovable.app)

**Abstract.** Compare the three samplers on the same CSP and quantify when correlated inputs change the answer.

**Mirrors.** **Live MC** tab → *Use correlation* toggle (and the underlying sampler choice).


In [ ]:
# cubespec-bootstrap
# Install cubespec on first run (Colab/Binder safe; no-op if already importable).
try:
    import cubespec  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'git+https://github.com/ai-systems-today/cubespec.git'])
    import cubespec  # noqa: F401


## 1. Three samplers, one CSP

* **Independent** — every input drawn from its own marginal Gaussian.
* **Latin Hypercube (LHS)** — stratified independent samples; better space-filling.
* **Multivariate Normal (MVN)** — Cholesky factor of a correlation matrix encodes joint dependence.

We compare the three on the same N and report the resulting variance of P9 and the marginal coverage.


In [ ]:
import numpy as np
from cubespec import DEFAULT_CSP, sample_independent, sample_lhs, sample_mvn, compute_outputs_batch
from cubespec.correlation import GEOMETRY_PRESET

N = 5_000
X_i = sample_independent(DEFAULT_CSP, n=N, seed=1337)
X_l = sample_lhs(DEFAULT_CSP, n=N, seed=1337)
X_m = sample_mvn(DEFAULT_CSP, n=N, seed=1337, corr=GEOMETRY_PRESET)
for name, X in (('independent', X_i), ('lhs', X_l), ('mvn-geometry', X_m)):
    Y = compute_outputs_batch(X)[:, 2]
    print(f'{name:14s}  P9 mean = {Y.mean():.3f}  sd = {Y.std(ddof=1):.3f}')


Independent and LHS agree on the mean (same marginals); LHS gives a very slightly tighter sd because it suppresses clumping. The MVN result has different marginals once correlation is added — the spread is wider because the geometry inputs reinforce each other.


## 2. Coverage of the unit hypercube projection

Project each sampler onto the (P1_dx, P2_dy) plane.


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(11, 3.6), sharex=True, sharey=True)
for ax, (X, name, color) in zip(axes, (
    (X_i, 'independent', '#5b8def'),
    (X_l, 'lhs',         '#28a745'),
    (X_m, 'mvn-geom',    '#e94f64'),
)):
    ax.scatter(X[:, 1], X[:, 2], s=2, alpha=0.4, color=color)
    ax.set_title(name)
    ax.set_xlabel('P1_dx (mm)')
axes[0].set_ylabel('P2_dy (mm)')
fig.suptitle('Joint coverage on (P1_dx, P2_dy)', fontsize=11)
fig.tight_layout(); plt.show()


## 3. When to use each

* Independent — default; cheapest; correct when inputs are physically uncorrelated.
* LHS — when N is small and you cannot afford clumping (e.g. expensive surrogate).
* MVN — when inputs share known dependence (geometry tolerances, correlated forces).


In [ ]:
# cubespec-reproducibility-footer
import platform, sys, numpy as np, scipy, pandas as pd, cubespec
print('Reproducibility')
print('  cubespec :', cubespec.__version__)
print('  python   :', sys.version.split()[0], 'on', platform.system())
print('  numpy    :', np.__version__)
print('  scipy    :', scipy.__version__)
print('  pandas   :', pd.__version__)
